In [1]:
import pandas as pd

week1_tracking = pd.read_csv("Data/week1_tracking_enhanced.csv")
players = pd.read_csv("/Users/alecxszhang/Desktop/Stat 390/Data/players.csv")
plays = pd.read_csv("/Users/alecxszhang/Desktop/Stat 390/Data/plays.csv")
player_play = pd.read_csv("/Users/alecxszhang/Desktop/Stat 390/Data/player_play.csv")
games = pd.read_csv("/Users/alecxszhang/Desktop/Stat 390/Data/games.csv")

In [2]:
week1_tracking["playId"]

0            64
1            64
2            64
3            64
4            64
           ... 
7200336    3696
7200337    3696
7200338    3696
7200339    3696
7200340    3696
Name: playId, Length: 7200341, dtype: int64

In [3]:
import re

import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Initial Cleaning (Drop NaNs)
df_clean = week1_tracking.dropna(subset=['pff_passCoverage', 'pff_manZone']).copy()


# 1. Collapse coverage families, regardless of punctuation / spacing variants
def collapse_coverage_label(value):
    if pd.isna(value):
        return value

    raw_value = str(value).strip()
    normalized = re.sub(r"[^a-z0-9]+", " ", raw_value.lower()).strip()

    if normalized.startswith("cover 0"):
        return "Cover 0"
    if normalized.startswith("cover 1"):
        return "Cover 1"
    if normalized.startswith("cover 2") or normalized == "2 man":
        return "Cover 2"
    if normalized.startswith("cover 3"):
        return "Cover 3"
    if normalized.startswith("cover 4") or normalized == "quarters":
        return "Quarters/C4"
    if normalized.startswith("cover 6"):
        return "Cover 6"
    if normalized in {"red zone", "goal line", "miscellaneous"}:
        return "Exotic/GoalLine"
    if normalized == "prevent":
        return "Prevent"
    if normalized == "bracket":
        return "Bracket"

    return raw_value


# 2. Apply the collapsing function
df_clean['pff_passCoverage_collapsed'] = df_clean['pff_passCoverage'].map(collapse_coverage_label)

# 3. Create play-level labels using the collapsed column
play_level_labels = (
    df_clean
    .groupby('playId')[['pff_passCoverage_collapsed', 'pff_manZone']]
    .first()
    .reset_index()
)

# 4. Build stratify key
play_level_labels['stratify_key'] = (
    play_level_labels['pff_passCoverage_collapsed'].astype(str)
    + "_"
    + play_level_labels['pff_manZone'].astype(str)
)

# 5. Drop singleton classes (can't stratify-split with only 1 member)
class_counts = play_level_labels['stratify_key'].value_counts()
valid_classes = class_counts[class_counts >= 2].index
play_level_labels = play_level_labels[play_level_labels['stratify_key'].isin(valid_classes)]

# 6. Stratified train/test split
train_ids, test_ids = train_test_split(
    play_level_labels['playId'],
    test_size=0.20,
    random_state=42,
    stratify=play_level_labels['stratify_key']
)

# 7. Map back to full tracking data
train_df = df_clean[df_clean['playId'].isin(train_ids)].copy()
test_df = df_clean[df_clean['playId'].isin(test_ids)].copy()

print(f"✅ Success! Collapsed to {play_level_labels['pff_passCoverage_collapsed'].nunique()} coverage types.")
print(f"   Dropped {len(class_counts) - len(valid_classes)} rare coverage combinations.")
print(f"   Training plays: {len(train_ids)} | Test plays: {len(test_ids)}")

✅ Success! Collapsed to 9 coverage types.
   Dropped 0 rare coverage combinations.
   Training plays: 1240 | Test plays: 310


In [4]:
week1_tracking.columns

Index(['gameId', 'playId', 'nflId', 'displayName', 'frameId', 'frameType',
       'time', 'jerseyNumber', 'club', 'playDirection', 'x', 'y', 's', 'a',
       'dis', 'o', 'dir', 'event', 'sideofball', 'quarter', 'down',
       'yardsToGo', 'gameClock', 'absoluteYardlineNumber', 'yardlineSide',
       'yardlineNumber', 'playClockAtSnap', 'preSnapHomeScore',
       'preSnapVisitorScore', 'preSnapHomeTeamWinProbability',
       'preSnapVisitorTeamWinProbability', 'expectedPoints', 'possessionTeam',
       'defensiveTeam', 'offenseFormation', 'receiverAlignment',
       'pff_passCoverage', 'pff_manZone'],
      dtype='str')

In [5]:
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import time
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

DATA_DIR = Path("Data")
TRACKING_FILES = sorted(DATA_DIR.glob("week*_tracking_enhanced.csv"))

FEATURE_COLUMNS = [
    "x", "y", "s", "a", "dis", "o", "dir", "event", "sideofball",
    "quarter", "down", "yardsToGo", "gameClock", "absoluteYardlineNumber",
    "yardlineSide", "yardlineNumber", "playClockAtSnap", "preSnapHomeScore",
    "preSnapVisitorScore", "preSnapHomeTeamWinProbability",
    "preSnapVisitorTeamWinProbability", "expectedPoints", "possessionTeam",
    "defensiveTeam", "offenseFormation", "receiverAlignment",
]

TARGET_COLUMNS = ["pff_passCoverage_collapsed", "pff_manZone"]
RAW_TARGET_COLUMNS = ["pff_passCoverage", "pff_manZone"]
ID_COLUMNS = ["gameId", "playId", "frameId", "frameType"]


In [6]:
def game_clock_to_seconds(value):
    if pd.isna(value):
        return np.nan
    try:
        minutes, seconds = str(value).split(":")
        return int(minutes) * 60 + int(seconds)
    except ValueError:
        return np.nan


def load_baseline_frame(files: Iterable[Path]) -> pd.DataFrame:
    usecols = ID_COLUMNS + FEATURE_COLUMNS + RAW_TARGET_COLUMNS
    frames = []

    for path in files:
        weekly = pd.read_csv(path, usecols=usecols, low_memory=False)
        weekly = weekly.loc[weekly["frameType"] == "BEFORE_SNAP"].copy()
        weekly = weekly.sort_values(["gameId", "playId", "frameId"])
        weekly = weekly.groupby(["gameId", "playId"], group_keys=False).tail(1)
        frames.append(weekly)

    df = pd.concat(frames, ignore_index=True)
    df["gameClock"] = df["gameClock"].map(game_clock_to_seconds)
    df["pff_passCoverage_collapsed"] = df["pff_passCoverage"].map(collapse_coverage_label)
    df["play_key"] = df["gameId"].astype(str) + "_" + df["playId"].astype(str)
    return df


baseline_df = load_baseline_frame(TRACKING_FILES)

print(f"Loaded {len(TRACKING_FILES)} weekly files")
print(f"Rows in baseline set: {len(baseline_df):,}")
print(f"Collapsed coverage labels available: {baseline_df['pff_passCoverage_collapsed'].nunique()}")
display(baseline_df.head())


Loaded 9 weekly files
Rows in baseline set: 16,119
Collapsed coverage labels available: 9


,gameId,playId,frameId,frameType,x,y,s,a,dis,o,dir,event,sideofball,quarter,down,yardsToGo,gameClock,absoluteYardlineNumber,yardlineSide,yardlineNumber,playClockAtSnap,preSnapHomeScore,preSnapVisitorScore,preSnapHomeTeamWinProbability,preSnapVisitorTeamWinProbability,expectedPoints,possessionTeam,defensiveTeam,offenseFormation,receiverAlignment,pff_passCoverage,pff_manZone,pff_passCoverage_collapsed,play_key
0,2022090800,56,145,BEFORE_SNAP,85.160004,29.690001,0.0,0.00,0.00,NaN,NaN,NaN,NaN,1,1,10,900,85,BUF,25,12.0,0,0,0.413347,0.586653,1.298699,BUF,LA,SHOTGUN,2x2,Cover 6-Left,Zone,Cover 6,2022090800_56
1,2022090800,80,87,BEFORE_SNAP,79.330002,29.520000,0.0,0.00,0.00,NaN,NaN,NaN,NaN,1,2,4,869,79,BUF,31,16.0,0,0,0.413316,0.586684,1.303119,BUF,LA,EMPTY,3x2,Cover 6-Left,Zone,Cover 6,2022090800_80
2,2022090800,101,105,BEFORE_SNAP,72.040001,29.520000,0.0,0.08,0.01,NaN,NaN,NaN,NaN,1,1,10,834,72,BUF,38,13.0,0,0,0.399819,0.600181,2.126690,BUF,LA,I_FORM,2x1,Cover-6 Right,Zone,Cover 6,2022090800_101
3,2022090800,122,112,BEFORE_SNAP,65.400002,29.480000,0.0,0.00,0.00,NaN,NaN,NaN,NaN,1,2,3,795,65,BUF,45,9.0,0,0,0.384969,0.615031,2.689053,BUF,LA,SHOTGUN,2x1,Cover-3,Zone,Cover 3,2022090800_122
4,2022090800,167,101,BEFORE_SNAP,57.509998,23.889999,0.0,0.00,0.00,NaN,NaN,NaN,NaN,1,2,8,714,57,LA,47,9.0,0,0,0.387554,0.612446,2.574206,BUF,LA,EMPTY,3x2,Cover-3,Zone,Cover 3,2022090800_167


In [7]:
numeric_features = [
    "x", "y", "s", "a", "dis", "o", "dir", "quarter", "down", "yardsToGo",
    "gameClock", "absoluteYardlineNumber", "yardlineNumber", "playClockAtSnap",
    "preSnapHomeScore", "preSnapVisitorScore", "preSnapHomeTeamWinProbability",
    "preSnapVisitorTeamWinProbability", "expectedPoints",
]

categorical_features = [col for col in FEATURE_COLUMNS if col not in numeric_features]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numeric_features,
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_features,
        ),
    ]
)


In [ ]:
def evaluate_logistic_target(df: pd.DataFrame, target: str):
    model_df = df.dropna(subset=[target]).copy()
    X = model_df[FEATURE_COLUMNS]
    y = model_df[target]
    groups = model_df["play_key"]

    splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(splitter.split(X, y, groups=groups))

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]
    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    clf = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=1000,
            solver="saga",
            class_weight="balanced",
        )),
    ])

    fit_start = time.perf_counter()
    clf.fit(X_train, y_train)
    fit_seconds = time.perf_counter() - fit_start

    predict_start = time.perf_counter()
    preds = clf.predict(X_test)
    predict_seconds = time.perf_counter() - predict_start

    metrics = {
        "target": target,
        "train_rows": len(X_train),
        "test_rows": len(X_test),
        "num_classes": y.nunique(),
        "accuracy": accuracy_score(y_test, preds),
        "macro_f1": f1_score(y_test, preds, average="macro"),
        "weighted_f1": f1_score(y_test, preds, average="weighted"),
        "fit_seconds": fit_seconds,
        "predict_seconds": predict_seconds,
        "total_target_seconds": fit_seconds + predict_seconds,
    }

    return clf, metrics, y_test, preds


loop_start = time.perf_counter()

results = {}
metric_rows = []

for target in TARGET_COLUMNS:
    model, metrics, y_test, preds = evaluate_logistic_target(baseline_df, target)
    results[target] = {
        "model": model,
        "y_test": y_test,
        "predictions": preds,
    }
    metric_rows.append(metrics)

loop_seconds = time.perf_counter() - loop_start
print(f"Total baseline loop runtime: {loop_seconds:.2f} seconds")

metrics_df = pd.DataFrame(metric_rows).sort_values("macro_f1", ascending=False)
display(metrics_df)


In [9]:
for target in TARGET_COLUMNS:
    print(f"\n===== {target} =====")
    print(classification_report(
        results[target]["y_test"],
        results[target]["predictions"],
        zero_division=0
    ))



===== pff_passCoverage_collapsed =====
                 precision    recall  f1-score   support

        Bracket       0.04      0.47      0.07        15
        Cover 0       0.16      0.42      0.24       115
        Cover 1       0.35      0.22      0.27       685
        Cover 2       0.31      0.39      0.34       440
        Cover 3       0.56      0.26      0.35      1133
        Cover 6       0.21      0.40      0.27       250
Exotic/GoalLine       0.39      0.71      0.50       132
        Prevent       0.11      0.64      0.19        14
    Quarters/C4       0.27      0.26      0.26       402

       accuracy                           0.31      3186
      macro avg       0.27      0.42      0.28      3186
   weighted avg       0.39      0.31      0.32      3186


===== pff_manZone =====
              precision    recall  f1-score   support

         Man       0.39      0.48      0.43       846
       Other       0.29      0.80      0.42       161
        Zone       0.83     